# Trabajo final

Objetivo

- Predecir señales de crisis de mora usando una CNN 1D sobre series históricas mensuales.

Entradas

- Datos mensuales agregados por bloque (mes, riesgo, sector, provincia, sucursal): número de créditos, montos, mora, tasas, y métricas adicionales (concentración, variabilidad, antigüedad).

Salidas

- Modelo CNN que entrega probabilidades de eventos (crisis) para 18 horizontes (horizonte_1 ... horizonte_18).
- Artefactos: modelo entrenado, scaler, archivo CSV para dashboard y archivo JSON de configuración.

Pasos principales

1. Extraer y agregar datos desde PostgreSQL.
2. Crear features y métricas predictivas.
3. Construir secuencias (ventana histórica = 6 meses) para la CNN.
4. Normalizar características y separar train/val/test.
5. Diseñar y entrenar el modelo CNN multi-output (18 salidas).
6. Evaluar, guardar modelos y artefactos.
7. Desplegar/usar la app Streamlit para predicciones y visualización.

Artefactos guardados

- `modelos_cnn/modelo_cnn_multi_18m.h5`
- `modelos_cnn/best_model_multi_18m.h5`
- `modelos_cnn/scaler_multi_18m.pkl`
- `modelos_cnn/datos_dashboard_multi_18m.csv`
- `modelos_cnn/config_18m.json`

Requisitos (rápido)

- Python 3.8+ con: `pandas`, `numpy`, `scikit-learn`, `tensorflow`, `sqlalchemy`, `psycopg2-binary`, `plotly`, `streamlit`.

Instrucciones rápidas

- Ejecutar las celdas en orden: extracción → features → secuencias → entrenamiento → evaluación → guardado → app.
- Si hay errores de shapes o métricas, ejecutar la celda de validación y ajustar inputs antes de entrenar.


## Desarrollo

### Instalación de librerías

In [ ]:
!pip install psycopg2-binary pandas numpy tensorflow scikit-learn plotly streamlit seaborn sqlalchemy psycopg2-binary pandas numpy tensorflow scikit-learn plotly streamlit seaborn sqlalchemy

### Importar librerías

In [ ]:
import joblib
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import tensorflow as tf
import warnings

In [ ]:
from datetime import datetime, timedelta
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sqlalchemy import create_engine
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

### Configurar librerías

In [ ]:
# Configurar filtro de errores
warnings.filterwarnings('ignore')

# Configuración de visualizaciones
plt.style.use('seaborn-v0_8-deep')
sns.set_palette("husl")
sns.husl_palette(as_cmap=True)
%matplotlib inline

### Conectar Base de Datos -> PostgreSQL

In [ ]:
DB_CONFIG = {
    'host': 'localhost',
    'port': '5432',
    'database': 'analisis_db',
    'user': 'usuario',
    'password': 'mi_clave_segura'
}

# Crear engine de SQLAlchemy
connection_string = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
engine = create_engine(connection_string)

### Probar la conexión a la base de datos

Se crea una función para ejecutar la consulta SQL

In [ ]:
def ejecutar_query(query, descripcion="", chunksize=None):
    """
    Ejecuta una query SQL y retorna un DataFrame
    Si chunksize se especifica, procesa en lotes para datasets grandes
    """
    try:
        if chunksize:
            print(f"📊 Procesando {descripcion} en lotes de {chunksize:,} registros...")
            dfs = []
            for chunk in pd.read_sql_query(query, engine, chunksize=chunksize):
                dfs.append(chunk)
                print(f"  ✅ Lote procesado: {len(chunk):,} registros (total acumulado: {sum(len(df) for df in dfs):,})")
            df = pd.concat(dfs, ignore_index=True)
        else:
            df = pd.read_sql_query(query, engine)
        
        print(f"✅ {descripcion}: {len(df):,} registros obtenidos")
        return df
    except Exception as e:
        print(f"❌ Error en query {descripcion}: {e}")
        return None


SQL para probar si hay conexión y una visión previa de créditos

In [ ]:
test_query = """
select count(*) as total_prestamos
from 
	cabecera_prestamos cp,
	amortizacal_prestamos ap
where cp.numero_credito = ap.numero_credito
and  ap.fecinical::DATE BETWEEN '2015-01-01'::DATE AND '2025-06-30'::DATE
"""

Ejecutar test de conexión a la base de datos.

In [ ]:
test_result = ejecutar_query(test_query, "Test de conexión")
if test_result is not None:
    print(f"📊 Total préstamos desde enero 2024: {test_result['total_prestamos'].iloc[0]:,}")

## Consulta para extraer los datos principales para análisis de mora

In [ ]:
# Query principal para extraer datos agregados por mes-riesgo-sector
query_bloques_18m = """
WITH datos_mensuales AS (
    SELECT 
        DATE_TRUNC('month', cp.fecha_credito) as mes,
        COALESCE(cp.codigo_riesgo, 'SIN_RIESGO') as riesgo,
        COALESCE(cp.act_economica_nvl1, 'SIN_SECTOR') as sector,
        cp.codigo_provincia,
        cp.codigo_sucursal,
        
        -- Métricas de volumen
        COUNT(*) as num_creditos,
        SUM(cp.monto_acreditado) as monto_total,
        AVG(cp.monto_acreditado) as monto_promedio,
        
        -- Métricas de morosidad
        AVG(cp.tot_dias_mora) as dias_mora_promedio,
        AVG(cp.tot_num_moras) as num_moras_promedio,
        COUNT(CASE WHEN cp.tot_dias_mora > 90 THEN 1 END) as creditos_mora_90,
        
        -- Métricas de riesgo
        COUNT(CASE WHEN cp.judicial = 'S' THEN 1 END) as creditos_judiciales,
        SUM(cp.gestion_cobro) as total_gestion_cobro,
        SUM(cp.costo_judicial) as total_costo_judicial,
        
        -- Métricas financieras
        AVG(cp.tasa_interes) as tasa_interes_promedio,
        AVG(COALESCE(cp.saldo_capital, 0)) as saldo_promedio,
        COUNT(CASE WHEN cp.estado_cred IN ('C', 'L') THEN 1 END) as creditos_cerrados,
        
        -- NUEVAS MÉTRICAS PREDICTIVAS
        -- 2. Concentración de riesgo (clientes únicos y créditos por cliente)
        COUNT(DISTINCT cp.codigo_socio) as num_clientes_unicos,
        
        -- 3. Estacionalidad (mes del año)
        EXTRACT(MONTH FROM cp.fecha_credito) as mes_del_ano,
        
        -- 4. Duración promedio de créditos (plazo)
        AVG(cp.num_cuotas) as plazo_promedio,
        
        -- 5. Variabilidad de montos
        STDDEV(cp.monto_acreditado) as desviacion_montos,
        
        -- 6. Antigüedad de cartera
        AVG(EXTRACT(MONTH FROM AGE(CURRENT_DATE, cp.fecha_credito))) as antiguedad_promedio_meses
        
    FROM cabecera_prestamos cp
    WHERE cp.fecha_credito >= '2015-07-01' 
      AND cp.fecha_credito < '2025-07-01'
    GROUP BY DATE_TRUNC('month', cp.fecha_credito), cp.codigo_riesgo, cp.act_economica_nvl1, 
             cp.codigo_provincia, cp.codigo_sucursal, EXTRACT(MONTH FROM cp.fecha_credito)
),
datos_con_lag AS (
    SELECT 
        *,
        -- 1. Ratio de crecimiento del sector (créditos mes anterior)
        LAG(num_creditos, 1) OVER (
            PARTITION BY riesgo, sector 
            ORDER BY mes
        ) as num_creditos_mes_anterior,
        -- Ratio de crecimiento de monto
        LAG(monto_total, 1) OVER (
            PARTITION BY riesgo, sector 
            ORDER BY mes
        ) as monto_mes_anterior
    FROM datos_mensuales
)
SELECT 
    mes,
    riesgo,
    sector,
    codigo_provincia,
    codigo_sucursal,
    num_creditos,
    monto_total,
    monto_promedio,
    dias_mora_promedio,
    num_moras_promedio,
    
    -- Ratios calculados (usar ::numeric para ROUND)
    ROUND((creditos_mora_90::numeric / NULLIF(num_creditos, 0)) * 100, 2) as tasa_mora_90,
    ROUND((creditos_judiciales::numeric / NULLIF(num_creditos, 0)) * 100, 2) as tasa_judicial,
    ROUND((creditos_cerrados::numeric / NULLIF(num_creditos, 0)) * 100, 2) as tasa_cierre,
    
    total_gestion_cobro,
    total_costo_judicial,
    tasa_interes_promedio,
    saldo_promedio,
    creditos_cerrados,
    
    -- NUEVAS MÉTRICAS PREDICTIVAS
    -- Concentración de riesgo
    num_clientes_unicos,
    ROUND(num_creditos::numeric / NULLIF(num_clientes_unicos, 0), 2) as creditos_por_cliente,
    
    -- Estacionalidad
    mes_del_ano,
    
    -- Plazo promedio
    ROUND(plazo_promedio::numeric, 2) as plazo_promedio,
    
    -- Variabilidad de montos
    ROUND(desviacion_montos::numeric, 2) as desviacion_montos,
    ROUND((desviacion_montos::numeric / NULLIF(monto_promedio, 0)) * 100, 2) as coef_variacion_montos,
    
    -- Antigüedad de cartera
    ROUND(antiguedad_promedio_meses::numeric, 2) as antiguedad_promedio_meses,
    
    -- Ratio de crecimiento
    num_creditos_mes_anterior,
    ROUND(((num_creditos::numeric - COALESCE(num_creditos_mes_anterior, num_creditos)) / 
           NULLIF(num_creditos_mes_anterior, 0)) * 100, 2) as tasa_crecimiento_creditos,
    monto_mes_anterior,
    ROUND(((monto_total::numeric - COALESCE(monto_mes_anterior, monto_total)) / 
           NULLIF(monto_mes_anterior, 0)) * 100, 2) as tasa_crecimiento_monto
    
FROM datos_con_lag
WHERE num_creditos >= 10  -- Filtrar bloques con pocos créditos
ORDER BY mes, riesgo, sector
"""

### Ejecutar consulta Principal

In [ ]:
df_bloques_18m = ejecutar_query(query_bloques_18m, "Datos agregados 10 años (Jul 2015 - Jun 2025)", chunksize=100000)

if df_bloques_18m is not None:
    print(f"📊 Período: {df_bloques_18m['mes'].min()} a {df_bloques_18m['mes'].max()}")
    print(f"📊 Bloques únicos: {len(df_bloques_18m)}")
    print(f"📊 Sectores únicos: {df_bloques_18m['sector'].nunique()}")
    print(f"📊 Tipos de riesgo: {df_bloques_18m['riesgo'].nunique()}")
    print(f"📊 Sucursales únicas: {df_bloques_18m['codigo_sucursal'].nunique()}")
    print(f"📊 Nuevas métricas disponibles: plazo_promedio, creditos_por_cliente, tasa_crecimiento_creditos, etc.")
    
    # Mostrar primeros registros
    display(df_bloques_18m.head())

## Análisis Exploratorio de Datos (EDA, por sus siglas en inglés) 

In [ ]:
print(df_bloques_18m.describe())

### DISTRIBUCIÓN POR SECTOR

In [ ]:
sector_summary = df_bloques_18m.groupby('sector').agg({
    'num_creditos': 'sum',
    'monto_total': 'sum',
    'tasa_mora_90': 'mean',
    'tasa_judicial': 'mean',
    'plazo_promedio': 'mean',
    'creditos_por_cliente': 'mean',
    'tasa_crecimiento_creditos': 'mean'
}).round(2)
display(sector_summary)

### DISTRIBUCIÓN POR RIESGO 

In [ ]:
riesgo_summary = df_bloques_18m.groupby('riesgo').agg({
    'num_creditos': 'sum',
    'monto_total': 'sum',
    'tasa_mora_90': 'mean',
    'tasa_judicial': 'mean',
    'plazo_promedio': 'mean',
    'creditos_por_cliente': 'mean',
    'tasa_crecimiento_creditos': 'mean'
}).round(2)
display(riesgo_summary)

### NUEVAS MÉTRICAS PREDICTIVAS

In [ ]:
nuevas_metricas = ['plazo_promedio', 'creditos_por_cliente', 'desviacion_montos', 
                   'coef_variacion_montos', 'antiguedad_promedio_meses', 
                   'tasa_crecimiento_creditos', 'tasa_crecimiento_monto', 'mes_del_ano']
print(df_bloques_18m[nuevas_metricas].describe())

### Creacion de Caracteristicas o Features

In [ ]:
# Crear DataFrame base para series temporales
df_features = df_bloques_18m.copy()

# Convertir fecha a datetime
df_features['mes'] = pd.to_datetime(df_features['mes'])

# Crear identificador único por bloque (incluyendo sucursal)
df_features['bloque_id'] = df_features['riesgo'] + '_' + df_features['sector'] + '_' + df_features['codigo_sucursal'].astype(str)

# Rellenar valores NaN en las nuevas métricas
df_features['tasa_crecimiento_creditos'] = df_features['tasa_crecimiento_creditos'].fillna(0)
df_features['tasa_crecimiento_monto'] = df_features['tasa_crecimiento_monto'].fillna(0)
df_features['num_creditos_mes_anterior'] = df_features['num_creditos_mes_anterior'].fillna(df_features['num_creditos'])
df_features['monto_mes_anterior'] = df_features['monto_mes_anterior'].fillna(df_features['monto_total'])
df_features['desviacion_montos'] = df_features['desviacion_montos'].fillna(0)
df_features['coef_variacion_montos'] = df_features['coef_variacion_montos'].fillna(0)

# Seleccionar features principales (incluyendo nuevas métricas predictivas)
features_numericas = [
    # Métricas originales
    'num_creditos', 'monto_promedio', 'dias_mora_promedio', 
    'tasa_mora_90', 'tasa_judicial', 'tasa_cierre',
    'total_gestion_cobro', 'tasa_interes_promedio',
    # Nuevas métricas predictivas
    'creditos_por_cliente',          # Concentración de riesgo
    'mes_del_ano',                   # Estacionalidad
    'plazo_promedio',                # Duración promedio
    'desviacion_montos',             # Variabilidad
    'coef_variacion_montos',         # Coeficiente de variación
    'antiguedad_promedio_meses',     # Antigüedad de cartera
    'tasa_crecimiento_creditos',     # Ratio de crecimiento
    'tasa_crecimiento_monto'         # Ratio de crecimiento de monto
]

print(f"✅ Features seleccionadas: {len(features_numericas)}")
print(f"✅ Bloques únicos: {df_features['bloque_id'].nunique()}")
print(f"✅ Rango temporal: {df_features['mes'].min()} a {df_features['mes'].max()}")
print(f"✅ Nuevas features predictivas añadidas: creditos_por_cliente, mes_del_ano, plazo_promedio, etc.")

### Función para señalar las posibles crisis en la recuperación de cartera

Estas son:

- tasa de mora sobre 90 días
- alta rotación
- tasa de crecimiento de créditos
- coeficiente de variación de montos
- plazos largos con alta mora, lo que expone una mayor exposición
- antigüedad de la cartera

In [ ]:
def calcular_crisis_flag_mejorado(row):
    """
    Crisis más robusta usando puntuación ponderada con nuevas métricas predictivas
    """
    crisis_score = 0
    
    # Ponderaciones según importancia - Métricas originales
    if row['tasa_mora_90'] > 15:
        crisis_score += 3  # Más peso a mora >90 días
    
    if row['tasa_judicial'] > 5:
        crisis_score += 2
    
    if row['dias_mora_promedio'] > 45:  # Más estricto (antes 30)
        crisis_score += 2
    
    if row['total_gestion_cobro'] > row['monto_total'] * 0.08:  # 8% (antes 5%)
        crisis_score += 1
    
    # Indicadores de rotación
    if row['creditos_cerrados'] / row['num_creditos'] > 0.3:  # Alta rotación
        crisis_score += 1
    
    if row['num_creditos'] < 50 and row['tasa_mora_90'] > 20:  # Pequeños bloques con alta mora
        crisis_score += 2
    
    # NUEVOS INDICADORES PREDICTIVOS
    # Concentración de riesgo (muchos créditos por cliente = mayor riesgo)
    if row['creditos_por_cliente'] > 3:
        crisis_score += 1
    
    # Crecimiento negativo sostenido (señal de problemas)
    if row['tasa_crecimiento_creditos'] < -20:
        crisis_score += 1
    
    # Alta variabilidad en montos (inestabilidad)
    if row['coef_variacion_montos'] > 100:
        crisis_score += 1
    
    # Plazos muy largos con alta mora (mayor exposición)
    if row['plazo_promedio'] > 36 and row['tasa_mora_90'] > 10:
        crisis_score += 1
    
    # Cartera muy antigua con problemas
    if row['antiguedad_promedio_meses'] > 60 and row['tasa_judicial'] > 3:
        crisis_score += 1
    
    # Crisis si score >= 4 (más robusto)
    return 1 if crisis_score >= 4 else 0

# Aplicar función de crisis
df_features['crisis_flag'] = df_features.apply(calcular_crisis_flag_mejorado, axis=1)

La función de las secuencias para CNN que analice múltiples horizontes.

Además, esta función debe ser eficiente en memoria

In [ ]:
# Preparar datos para CNN: crear secuencias temporales para múltiples horizontes
def crear_secuencias_cnn_multi(df, bloque_id, features, target, ventana=6, max_horizonte=18):
    """
    Crea secuencias temporales para CNN con múltiples horizontes de predicción
    ventana: número de meses históricos a usar
    max_horizonte: máximo horizonte de predicción (1-18 meses)
    """
    # Filtrar y ordenar por bloque específico
    df_bloque = df[df['bloque_id'] == bloque_id].sort_values('mes')
    
    if len(df_bloque) < ventana + max_horizonte:
        return None, None
    
    X_sequences = []
    y_sequences = []
    
    for i in range(len(df_bloque) - ventana - max_horizonte + 1):
        # Secuencia de entrada (ventana meses)
        X_seq = df_bloque[features].iloc[i:i+ventana].values
        
        # Targets para cada horizonte (1 a max_horizonte meses adelante)
        y_seq = []
        for h in range(1, max_horizonte + 1):
            if i + ventana + h - 1 < len(df_bloque):
                y_val = df_bloque[target].iloc[i + ventana + h - 1]
                y_seq.append(y_val)
            else:
                y_seq.append(0)  # Padding si no hay suficientes datos
        
        X_sequences.append(X_seq)
        y_sequences.append(y_seq)
    
    return np.array(X_sequences), np.array(y_sequences)


### Generar secuencias para múltiples horizontes

In [ ]:
X_all = []
y_all = []
bloques_validos = []

VENTANA_CNN = 6  # 6 meses de historia
MAX_HORIZONTE = 18  # Predecir hasta 18 meses adelante

for bloque in df_features['bloque_id'].unique():
    X_seq, y_seq = crear_secuencias_cnn_multi(
        df_features, bloque, features_numericas, 'crisis_flag', 
        VENTANA_CNN, MAX_HORIZONTE
    )
    
    if X_seq is not None and len(X_seq) > 0:
        X_all.extend(X_seq)
        y_all.extend(y_seq)
        bloques_validos.append(bloque)

X_cnn = np.array(X_all)
y_cnn = np.array(y_all)

print(f"✅ Secuencias creadas: {len(X_cnn)}")
print(f"✅ Shape entrada CNN: {X_cnn.shape}")
print(f"✅ Shape targets: {y_cnn.shape} (18 horizontes)")
print(f"✅ Bloques válidos: {len(bloques_validos)}")
print(f"✅ Balance target horizonte 1: {np.bincount(y_cnn[:, 0])}")

### Normalizar características

In [ ]:
# Normalizar features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_cnn.reshape(-1, X_cnn.shape[-1])).reshape(X_cnn.shape)

# Split temporal (importante: no mezclar temporalmente)
split_idx = int(len(X_scaled) * 0.7)
X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]

# Separar targets por horizonte
y_train_list = [y_cnn[:split_idx, i] for i in range(MAX_HORIZONTE)]
y_test_list = [y_cnn[split_idx:, i] for i in range(MAX_HORIZONTE)]

print(f"✅ Train set: {X_train.shape}")
print(f"✅ Test set: {X_test.shape}")
print(f"✅ Targets por horizonte: {len(y_train_list)} horizontes")

## Preparando modelo multi-output (corrección de métricas)

El número de horizontes define los límites de predicción

In [ ]:
# Asegurar que NUM_HORIZONTES y input_shape están definidos
NUM_HORIZONTES = MAX_HORIZONTE
input_shape = X_train.shape[1:]

## 🧠 Creación del modelo CNN para múltiples salidas de análisis

Se diseña una arquitectura basada en Conv1D para pronóstico multi-horizonte

In [ ]:
def crear_modelo_cnn_multioutput(input_shape, num_horizontes):
    """Crea modelo CNN para predicción multi-horizonte"""

    inputs = layers.Input(shape=input_shape)

    # Capas convolucionales
    x = layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv1D(filters=128, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Flatten()(x)

    # Capas densas
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    # Capas de salida para cada horizonte
    outputs = []
    for i in range(num_horizontes):
        output = layers.Dense(1, activation='sigmoid', name=f'horizonte_{i+1}')(x)
        outputs.append(output)

    # Modelo
    model = Model(inputs=inputs, outputs=outputs)

    # Compilar con métricas por salida (cada salida con sus métricas nombradas)
    metrics_dict = {f'horizonte_{i+1}': [
        'accuracy',
        tf.keras.metrics.Precision(name=f'precision_{i+1}'),
        tf.keras.metrics.Recall(name=f'recall_{i+1}')
    ] for i in range(num_horizontes)}

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=metrics_dict
    )

    return model

# Crear el modelo corregido
modelo_cnn = crear_modelo_cnn_multioutput(input_shape, NUM_HORIZONTES)
modelo_cnn.summary()

### Representación gráfica

> Prompt recomendado

- Descripción: muestra la evolución de las métricas durante el entrenamiento
- Mantiene la dimensión temporal y plotea por outputs/horizontes

![](arquitectura.png)

## Validación de dimensiones (shapes) y la consistencia del entrenamiento de múltiples horizontes

In [ ]:
# Información básica
n_model_outputs = len(modelo_cnn.outputs)
print(f"Salidas del modelo: {n_model_outputs}")
print("Nombres de las salidas:")
for o in modelo_cnn.outputs:
    print("  -", o.name)

In [ ]:
# Comprobar y normalizar listas de targets
print(f"NUM_HORIZONTES variable: {NUM_HORIZONTES}")
print(f"Longitud original y_train_list: {len(y_train_list)}")

# Convertir cada target a array 1D
for i in range(len(y_train_list)):
    arr = np.asarray(y_train_list[i])
    # eliminar dimensiones extra
    arr = arr.reshape(-1,)
    y_train_list[i] = arr

# aseguramos que cada uno de los target este en formato 1-D unidimencional
# evita errores con metricas de sklearn    
for i in range(len(y_test_list)):
    arr = np.asarray(y_test_list[i])
    y_test_list[i] = arr.reshape(-1,)
    
print("✔️ Targets convertidos a 1D arrays")

In [ ]:
# Verificar que la longitud de targets sea consistente y coincida con X_train
n_X = X_train.shape[0]
lengths = [arr.shape[0] for arr in y_train_list]
print(f"Muestras en X_train: {n_X}")
print(f"Longitudes de cada target (primeros 5): {lengths[:5]} (total {len(lengths)})")

In [ ]:
# Si hay mismatch, recortar a la longitud mínima
min_len = min([n_X] + lengths)
if min_len < n_X or any(l != n_X for l in lengths):
    print(f"⚠️ Mismatch detectado. Ajustando X_train y targets a min_len={min_len}")
    X_train = X_train[:min_len]
    y_train_list = [arr[:min_len] for arr in y_train_list]
    print("✔️ Ajuste aplicado")
else:
    print("✔️ Shapes consistentes entre X_train y targets")


In [ ]:
# Si el número de outputs del modelo no coincide con la longitud de targets, reconstruir modelo
if len(y_train_list) != n_model_outputs:
    new_horiz = len(y_train_list)
    print(f"⚠️ El modelo tiene {n_model_outputs} salidas pero hay {new_horiz} horizontes en targets.")
    print(f"Reconstruyendo el modelo con num_horizontes={new_horiz}...")
    NUM_HORIZONTES = new_horiz
    modelo_cnn = crear_modelo_cnn_multioutput(input_shape, NUM_HORIZONTES)
    print(f"✔️ Modelo reconstruido. Salidas ahora: {len(modelo_cnn.outputs)}")
else:
    print("✔️ Número de salidas del modelo coincide con targets")

In [ ]:
# Intentar un forward pass rápido (4 muestras) para validar la arquitectura
#
# Es ejecutar una pasada de inferencia (sin entrenamiento) con unas pocas muestras 
# para comprobar rápidamente que la arquitectura y los datos encajan
try:
    sample_in = X_train[:4]
    preds = modelo_cnn.predict(sample_in)
    print("Predict OK — número de arrays devueltos:", len(preds))
    print("Shapes de las primeras salidas:", [p.shape for p in preds[:5]])
except Exception as e:
    print("❌ Error durante predict de prueba:", e)

print("🔚 Validación completada. Si no hay errores, puedes ejecutar la celda de entrenamiento.")


### Creamos carpetas para el trabajo de aprendizaje

In [ ]:
# Asegurar carpeta para checkpoints
os.makedirs('modelos_cnn', exist_ok=True)

## 🏁 Creacion del modelo

Este punto es de lo mas importante pues obtendremos un modelo entreado para 18 horizontes de salida y requiere de trabajo de calculos extras para evitar errores previos al entrenamiento

In [ ]:
# Callbacks para Keras: control del entrenamiento
# EarlyStopping detiene el entrenamiento si la métrica 'val_loss' no mejora
# ModelCheckpoint guarda el mejor modelo obtenido; es útil para presentación posterior
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint('modelos_cnn/best_model_multi_18m.h5', monitor='val_loss', save_best_only=True)
    ## | este es el punto donde guardo mi modelo para presentacion posterior.
]

# Crear validación manual en lugar de validation_split para evitar errores
# al utilizar listas de targets (multi-output) con validation_split.
val_frac = 0.2
n_samples = X_train.shape[0]
val_size = int(n_samples * val_frac)

train_end = n_samples - val_size
X_train_final = X_train[:train_end]
X_val = X_train[train_end:]

y_train_final = [arr[:train_end] for arr in y_train_list]
y_val = [arr[train_end:] for arr in y_train_list]

historia = modelo_cnn.fit(
    X_train_final, y_train_final,
    validation_data=(X_val, y_val),
    epochs=100,  # Menos epochs para multioutput
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

## 📊 Evaluacion del modelo generado

In [ ]:
# Obtener predicciones
y_pred_proba_list = modelo_cnn.predict(X_test)
# Asegurarse que la primera salida existe
if not isinstance(y_pred_proba_list, (list, tuple)) or len(y_pred_proba_list) == 0:
    raise ValueError("La predicción del modelo no devolvió salidas como lista. Verifica la arquitectura del modelo.")

y_pred_1m = (y_pred_proba_list[0] > 0.5).astype(int).flatten()  # Horizonte 1 mes

# Métricas para horizonte 1
try:
    test_acc = accuracy_score(y_test_list[0], y_pred_1m)
    test_prec = precision_score(y_test_list[0], y_pred_1m, zero_division=0)
    test_recall = recall_score(y_test_list[0], y_pred_1m, zero_division=0)
except Exception as e:
    print("❌ Error calculando métricas (verifica shapes de y_test_list[0] y y_pred_1m):", e)
    raise

print(f"Accuracy: {test_acc:.4f}")
print(f"Precisión: {test_prec:.4f}")
print(f"Recall: {test_recall:.4f}")

if len(np.unique(y_test_list[0])) > 1:
    try:
        auc_score = roc_auc_score(y_test_list[0], y_pred_proba_list[0].flatten())
        print(f"AUC-ROC: {auc_score:.4f}")
    except Exception as e:
        print("⚠️ No se pudo calcular AUC-ROC:", e)

### Reportes finales

In [ ]:
print("\n📊 Reporte detallado:")
try:
    print(classification_report(y_test_list[0], y_pred_1m))
except Exception as e:
    print("❌ Error mostrando classification_report:", e)

print("\n📊 Matriz de confusión:")
try:
    print(confusion_matrix(y_test_list[0], y_pred_1m))
except Exception as e:
    print("❌ Error mostrando confusion_matrix:", e)


## Graficas Estadisticas sobre el entrenamiento

In [ ]:
# Gráfico de entrenamiento (soporta modelos multi-output)
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

hist = historia.history
epochs = range(1, len(hist.get('loss', [])) + 1)

# LOSS (siempre disponible)
ax1.plot(hist.get('loss', []), label='Train Loss', color='blue')
ax1.plot(hist.get('val_loss', []), label='Validation Loss', color='red')
ax1.set_title('Evolución de la Pérdida')
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# Helper para encontrar keys por métrica (puede haber keys por salida: e.g. 'horizonte_1_accuracy')
def keys_for(metric_name):
    keys = [k for k in hist.keys() if metric_name in k and not k.startswith('_') and not k.endswith('_samples')]
    # separar train/val por prefijo 'val_'
    train_keys = [k for k in keys if not k.startswith('val_')]
    val_keys = [k for k in keys if k.startswith('val_')]
    return train_keys, val_keys


def mean_series(keys):
    if not keys:
        return None
    arrs = [np.array(hist[k]) for k in keys]
    return np.mean(arrs, axis=0)

# ACCURACY: graficar mean si hay múltiples outputs, o la única si existe
train_acc_keys, val_acc_keys = keys_for('accuracy')
train_acc = mean_series(train_acc_keys)
val_acc = mean_series(val_acc_keys)
if train_acc is not None:
    ax2.plot(train_acc, label=('Train Accuracy' + ('' if len(train_acc_keys)==1 else ' (mean)')), color='blue')
    if val_acc is not None:
        ax2.plot(val_acc, label=('Validation Accuracy' + ('' if len(val_acc_keys)==1 else ' (mean)')), color='red')
else:
    ax2.text(0.5, 0.5, 'Accuracy no disponible en history', ha='center')
ax2.set_title('Evolución de la Precisión')
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

# PRECISION
train_prec_keys, val_prec_keys = keys_for('precision')
train_prec = mean_series(train_prec_keys)
val_prec = mean_series(val_prec_keys)
if train_prec is not None:
    ax3.plot(train_prec, label=('Train Precisión' + ('' if len(train_prec_keys)==1 else ' (mean)')), color='blue')
    if val_prec is not None:
        ax3.plot(val_prec, label=('Validation Precisión' + ('' if len(val_prec_keys)==1 else ' (mean)')), color='red')
else:
    ax3.text(0.5, 0.5, 'Precisión no disponible en history', ha='center')
ax3.set_title('Evolución de la Precisión')
ax3.set_xlabel('Época')
ax3.set_ylabel('Precisión')
ax3.legend()
ax3.grid(True)

# RECALL
train_rec_keys, val_rec_keys = keys_for('recall')
train_rec = mean_series(train_rec_keys)
val_rec = mean_series(val_rec_keys)
if train_rec is not None:
    ax4.plot(train_rec, label=('Train Recall' + ('' if len(train_rec_keys)==1 else ' (mean)')), color='blue')
    if val_rec is not None:
        ax4.plot(val_rec, label=('Validation Recall' + ('' if len(val_rec_keys)==1 else ' (mean)')), color='red')
else:
    ax4.text(0.5, 0.5, 'Recall no disponible en history', ha='center')
ax4.set_title('Evolución del Recall')
ax4.set_xlabel('Época')
ax4.set_ylabel('Recall')
ax4.legend()
ax4.grid(True)

plt.tight_layout()
plt.show()

### Analisis Grafico de las lineas del tiempo de Crisis en mora crediticia

In [ ]:
# Análisis temporal de crisis
crisis_temporal = df_features.groupby(['mes']).agg({
    'crisis_flag': ['sum', 'mean'],
    'tasa_mora_90': 'mean',
    'tasa_judicial': 'mean',
    'num_creditos': 'sum'
}).round(2)

crisis_temporal.columns = ['bloques_crisis', 'tasa_crisis', 'mora_90_prom', 'judicial_prom', 'total_creditos']

# Gráfico temporal
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Tasa de crisis temporal
ax1.plot(crisis_temporal.index, crisis_temporal['tasa_crisis'], marker='o', color='red', linewidth=2)
ax1.set_title('Evolución de la Tasa de Crisis (2015-2025)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Tasa de Crisis')
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Mora 90+ días
ax2.plot(crisis_temporal.index, crisis_temporal['mora_90_prom'], marker='s', color='orange', linewidth=2)
ax2.set_title('Evolución Mora >90 días (%)', fontsize=14, fontweight='bold')
ax2.set_ylabel('% Mora >90 días')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

# Tasa judicial
ax3.plot(crisis_temporal.index, crisis_temporal['judicial_prom'], marker='^', color='purple', linewidth=2)
ax3.set_title('Evolución Tasa Judicial (%)', fontsize=14, fontweight='bold')
ax3.set_ylabel('% Judicial')
ax3.grid(True, alpha=0.3)
ax3.tick_params(axis='x', rotation=45)

# Volumen total
ax4.bar(crisis_temporal.index, crisis_temporal['total_creditos'], color='skyblue', alpha=0.7)
ax4.set_title('Volumen Total de Créditos', fontsize=14, fontweight='bold')
ax4.set_ylabel('Número de Créditos')
ax4.tick_params(axis='x', rotation=45)

plt.suptitle('Panel de Crisis Energética 2015-2025', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## Preparar datos para modelo de presentación web.

- Guardamos el modelo
- Guardamos el scaler que contiene la información del transformador que normaliza o escala las características previas al entrenamiento.
- Datos para el dashboard
- Guardado de la configuración de la app web
  - Ventana de CNN: longitud de la secuencia histórica que el modelo usa como entrada.
  - Número de horizontes (18)
  - Número de características
  - Bloques válidos que se crearon.
  - Métricas finales para la presentación.

In [ ]:
# Guardar modelo CNN multioutput
modelo_cnn.save('modelos_cnn/modelo_cnn_multi_18m.h5')
print("💾 Modelo CNN Multioutput guardado")

# Guardar scaler
joblib.dump(scaler, 'modelos_cnn/scaler_multi_18m.pkl')
print("💾 Scaler guardado")

# Guardar datos para dashboard
df_features.to_csv('modelos_cnn/datos_dashboard_multi_18m.csv', index=False)
print("💾 Datos dashboard guardados")

# Guardar configuración
config_18m = {
    'ventana_cnn': VENTANA_CNN,
    'max_horizonte': MAX_HORIZONTE,
    'features_numericas': features_numericas,
    'bloques_validos': bloques_validos,
    'metricas_finales': {
        'accuracy': test_acc,
        'precision': test_prec,
        'recall': test_recall
    }
}

with open('modelos_cnn/config_18m.json', 'w') as f:
    json.dump(config_18m, f, indent=2, default=str)
print("💾 Configuración guardada")

> Para ejecutar el proyecto web, use el siguiente comando

```bash
streamlit run app_streamlit_18m_V2.py 
```

---
![](logo_site3.png)
<br>**Aprendizaje Profundo**<br>**Profesor**: Phd. Oscar Chang<br>**Alumnos**:<br>Omar Velez<br>Noe Sanchez<br>Ruben Basantes